# LeetCode 303 — Range Sum Query - Immutable bằng Python

**Link đề:** https://leetcode.com/problems/range-sum-query-immutable/
**Độ khó:** Easy · **Chủ đề:** Array, Design (thiết kế class), Prefix Sum (tổng tiền tố)

```python
class NumArray:

    def __init__(self, nums: list[int]):
        ...

    def sumRange(self, left: int, right: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu dạng bài **Design**: viết một **class** có hàm khởi tạo `__init__` và các method được gọi **nhiều lần**.
2. Nắm `self`, thuộc tính đối tượng, và cách LeetCode chạy chuỗi lệnh `["NumArray", "sumRange", ...]`.
3. Thấy vì sao cộng lại từ đầu cho mỗi truy vấn là chậm khi có nhiều truy vấn.
4. Xây dựng **mảng tổng tiền tố** (prefix sum) với phần tử canh `0` ở đầu, trả lời mỗi truy vấn trong `O(1)`.
5. Trace từng bước, chứng minh công thức `prefix[right + 1] - prefix[left]`, phân tích "tiền xử lý vs truy vấn".

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — trả lời nhiều truy vấn tổng đoạn

Cho mảng số nguyên `nums`. Cần xử lý **nhiều** truy vấn dạng: tính tổng các phần tử từ chỉ số `left` tới `right` (**bao gồm cả hai đầu**), với `left <= right`.

Cài đặt class `NumArray`:
- `NumArray(nums)` khởi tạo đối tượng với mảng `nums`.
- `sumRange(left, right)` trả về `nums[left] + nums[left + 1] + ... + nums[right]`.

Mảng **không thay đổi** sau khi khởi tạo (đó là ý nghĩa của "Immutable").

**Ví dụ:**
```text
Input
["NumArray", "sumRange", "sumRange", "sumRange"]
[[[-2, 0, 3, -5, 2, -1]], [0, 2], [2, 5], [0, 5]]
Output
[null, 1, -1, -3]
```

| Truy vấn | Các phần tử | Tổng |
|---|---|---:|
| `sumRange(0, 2)` | `-2 + 0 + 3` | `1` |
| `sumRange(2, 5)` | `3 + (-5) + 2 + (-1)` | `-1` |
| `sumRange(0, 5)` | cả mảng | `-3` |

**Constraints:** `1 <= nums.length <= 10^4`; `-10^5 <= nums[i] <= 10^5`; `0 <= left <= right < nums.length`; **tối đa `10^4` lần gọi** `sumRange`.

In [1]:
nums = [-2, 0, 3, -5, 2, -1]
print(nums[0] + nums[1] + nums[2])               # 1
print(sum(nums[2:6]))                            # -1: slice nums[2:6] lấy chỉ số 2..5
print(sum(nums))                                 # -3

1
-1
-3


## 2. Nền tảng Python cho bài này

### 2.1. Class, `__init__` và `self`

- **Class** là "khuôn" để tạo đối tượng. Mỗi đối tượng giữ **dữ liệu riêng** (thuộc tính) và có **hành vi** (method).
- `__init__` chạy **một lần** khi tạo đối tượng: `obj = NumArray(nums)`.
- `self` là chính đối tượng đó. Gán `self.data = ...` để lưu dữ liệu dùng lại ở các method sau.

In [2]:
class Counter:
    def __init__(self, start: int):
        self.value = start          # lưu vào đối tượng
        print("__init__ chạy, value =", self.value)

    def add(self, x: int) -> int:
        self.value += x             # dùng lại dữ liệu đã lưu
        return self.value

c = Counter(10)
print(c.add(5))     # 15
print(c.add(1))     # 16 — đối tượng "nhớ" giá trị giữa các lần gọi

__init__ chạy, value = 10
15
16


### 2.2. Slicing `nums[a:b]` và hàm `sum`

- `nums[a:b]` lấy các phần tử chỉ số `a, a+1, ..., b-1` (**không gồm `b`**).
- Muốn lấy đoạn **bao gồm** `right`, ta viết `nums[left:right + 1]`.
- `sum(list)` cộng tất cả phần tử — nhưng vẫn phải đi qua từng phần tử, tốn `O(độ dài)`.

In [3]:
nums = [-2, 0, 3, -5, 2, -1]
left, right = 2, 5
print(nums[left:right])          # [3, -5, 2] — THIẾU nums[5]
print(nums[left:right + 1])      # [3, -5, 2, -1] — đúng đoạn cần
print(sum(nums[left:right + 1])) # -1

[3, -5, 2]
[3, -5, 2, -1]
-1


### 2.3. Tổng tiền tố (prefix sum) là gì?

`prefix[i]` = tổng của **`i` phần tử đầu tiên** = `nums[0] + ... + nums[i - 1]`. Quy ước `prefix[0] = 0` (tổng của 0 phần tử).

Với `nums = [-2, 0, 3, -5, 2, -1]`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| `prefix[i]` | 0 | -2 | -2 | 1 | -4 | -2 | -3 |
| gồm | (rỗng) | -2 | -2,0 | -2,0,3 | …,-5 | …,2 | …,-1 |

Công thức xây dần: `prefix[i + 1] = prefix[i] + nums[i]`. Mảng prefix dài **`n + 1`**.

In [4]:
nums = [-2, 0, 3, -5, 2, -1]
prefix = [0] * (len(nums) + 1)          # [0, 0, 0, 0, 0, 0, 0]
for i in range(len(nums)):
    prefix[i + 1] = prefix[i] + nums[i]
print(prefix)                            # [0, -2, -2, 1, -4, -2, -3]

[0, -2, -2, 1, -4, -2, -3]


### 2.4. `itertools.accumulate` — prefix sum dựng sẵn

`accumulate([a, b, c])` sinh `a, a+b, a+b+c`. Tham số `initial=0` (Python 3.8+) thêm số `0` ở đầu — đúng mảng `prefix` của ta.

In [5]:
from itertools import accumulate
nums = [-2, 0, 3, -5, 2, -1]
print(list(accumulate(nums)))              # [-2, -2, 1, -4, -2, -3]
print(list(accumulate(nums, initial=0)))   # [0, -2, -2, 1, -4, -2, -3]

[-2, -2, 1, -4, -2, -3]
[0, -2, -2, 1, -4, -2, -3]


### 2.5. LeetCode chạy bài Design như thế nào?

Input gồm 2 list: tên lệnh và tham số. Lệnh đầu tạo đối tượng (output `null`), các lệnh sau gọi method. Ta viết hàm `run_ops` mô phỏng cách chấm để test giống hệt LeetCode.

In [6]:
def run_ops(cls, commands, args):
    out = []
    obj = None
    for cmd, arg in zip(commands, args):
        if cmd == cls.__name__ or obj is None:
            obj = cls(*arg)
            out.append(None)               # LeetCode in là null
        else:
            out.append(getattr(obj, cmd)(*arg))   # gọi method theo tên
    return out

print(run_ops(Counter, ["Counter", "add", "add"], [[10], [5], [1]]))

__init__ chạy, value = 10
[None, 15, 16]


## 3. Cách trực giác — cộng lại mỗi lần truy vấn

Lưu nguyên mảng. Mỗi lần `sumRange(left, right)` thì cộng các phần tử từ `left` tới `right`.

In [7]:
class NumArrayBrute:
    def __init__(self, nums: list[int]):
        self.nums = nums

    def sumRange(self, left: int, right: int) -> int:
        total = 0
        for i in range(left, right + 1):
            total += self.nums[i]
        return total

print(run_ops(NumArrayBrute, ["NumArrayBrute", "sumRange", "sumRange", "sumRange"],
              [[[-2, 0, 3, -5, 2, -1]], [0, 2], [2, 5], [0, 5]]))

[None, 1, -1, -3]


## 4. Nút thắt — công việc bị lặp lại giữa các truy vấn

- Mỗi truy vấn tốn `O(right - left + 1)`, worst case `O(n)`.
- Có `q` truy vấn → `O(q · n)`. Với `n = q = 10^4` là tới `10^8` phép cộng — **quá chậm** với Python.
- Quan sát: `sumRange(0, 4)` và `sumRange(0, 5)` cộng lại **gần như cùng các phần tử**. Ta đang lặp lại công việc.

Ta đếm số phép cộng để thấy rõ.

In [8]:
import random
random.seed(303)
n, q = 2000, 2000
nums_big = [random.randint(-10**5, 10**5) for _ in range(n)]
queries = [tuple(sorted((random.randrange(n), random.randrange(n)))) for _ in range(q)]

additions = sum(r - l + 1 for l, r in queries)
print(f"Brute force: khoảng {additions:,} phép cộng cho {q} truy vấn trên mảng {n} phần tử")
print(f"Prefix sum : {n:,} phép cộng để tiền xử lý + {q:,} phép trừ cho truy vấn")

Brute force: khoảng 1,315,475 phép cộng cho 2000 truy vấn trên mảng 2000 phần tử
Prefix sum : 2,000 phép cộng để tiền xử lý + 2,000 phép trừ cho truy vấn


### 4.1. Một ý tưởng "quá đà": tính sẵn tổng của mọi cặp `(left, right)`

Truy vấn sẽ `O(1)`, nhưng có `n(n+1)/2` cặp → `n = 10^4` cần ~`5·10^7` ô nhớ và thời gian tiền xử lý `O(n²)`. Không khả thi, nhưng gợi ý: **tính trước một phần thông tin** là hướng đúng — chỉ cần chọn thông tin **nhỏ gọn** hơn.

In [9]:
class NumArrayAllPairs:
    def __init__(self, nums: list[int]):
        n = len(nums)
        self.table = {}
        for l in range(n):
            running = 0
            for r in range(l, n):
                running += nums[r]
                self.table[(l, r)] = running

    def sumRange(self, left: int, right: int) -> int:
        return self.table[(left, right)]

obj = NumArrayAllPairs([-2, 0, 3, -5, 2, -1])
print(obj.sumRange(0, 2), obj.sumRange(2, 5), obj.sumRange(0, 5), "| số ô lưu:", len(obj.table))

1 -1 -3 | số ô lưu: 21


## 5. Insight — tổng đoạn = hiệu hai tổng tiền tố

```text
nums[0] + ... + nums[right]          = prefix[right + 1]
nums[0] + ... + nums[left - 1]       = prefix[left]
--------------------------------------------------- trừ đi
nums[left] + ... + nums[right]       = prefix[right + 1] - prefix[left]
```

Ví dụ `sumRange(2, 5)` với `prefix = [0, -2, -2, 1, -4, -2, -3]`:
`prefix[6] - prefix[2] = -3 - (-2) = -1` ✔.

**Vì sao cần `prefix[0] = 0`?** Khi `left = 0`, công thức cần `prefix[0]` = tổng của 0 phần tử = 0. Nhờ phần tử canh (sentinel) này, ta **không cần `if left == 0`**.

In [10]:
prefix = [0, -2, -2, 1, -4, -2, -3]
for left, right in [(0, 2), (2, 5), (0, 5), (3, 3)]:
    print(f"sumRange({left}, {right}) = prefix[{right + 1}] - prefix[{left}] = {prefix[right + 1]} - ({prefix[left]}) = {prefix[right + 1] - prefix[left]}")

sumRange(0, 2) = prefix[3] - prefix[0] = 1 - (0) = 1
sumRange(2, 5) = prefix[6] - prefix[2] = -3 - (-2) = -1
sumRange(0, 5) = prefix[6] - prefix[0] = -3 - (0) = -3
sumRange(3, 3) = prefix[4] - prefix[3] = -4 - (1) = -5


## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — trong `__init__`, tạo mảng dài `n + 1` toàn số 0:**
```python
self.prefix = [0] * (len(nums) + 1)
```

**Đoạn 2 — điền tổng tiền tố một lượt:**
```python
for i, x in enumerate(nums):
    self.prefix[i + 1] = self.prefix[i] + x
```
`enumerate(nums)` cho cặp `(chỉ số, giá trị)`.

**Đoạn 3 — trong `sumRange`, một phép trừ:**
```python
return self.prefix[right + 1] - self.prefix[left]
```

In [11]:
for i, x in enumerate([-2, 0, 3]):
    print("i =", i, "| x =", x)

i = 0 | x = -2
i = 1 | x = 0
i = 2 | x = 3


In [12]:
class NumArrayPrefix:
    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + x

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]

obj = NumArrayPrefix([-2, 0, 3, -5, 2, -1])
print(obj.prefix)
print(obj.sumRange(0, 2), obj.sumRange(2, 5), obj.sumRange(0, 5))

[0, -2, -2, 1, -4, -2, -3]
1 -1 -3


## 7. TRACE từng bước

### 7.1. Trace phần tiền xử lý

| i | x = nums[i] | prefix[i] | prefix[i+1] = prefix[i] + x |
|---:|---:|---:|---:|
| 0 | -2 | 0 | -2 |
| 1 | 0 | -2 | -2 |
| 2 | 3 | -2 | 1 |
| 3 | -5 | 1 | -4 |
| 4 | 2 | -4 | -2 |
| 5 | -1 | -2 | -3 |

In [13]:
def trace_build_prefix(nums: list[int]) -> list[int]:
    prefix = [0] * (len(nums) + 1)
    print("trạng thái đầu:", prefix)
    for i, x in enumerate(nums):
        prefix[i + 1] = prefix[i] + x
        print(f"i={i}, x={x:>2}: prefix[{i + 1}] = {prefix[i]:>2} + {x:>2} = {prefix[i + 1]:>2} -> {prefix}")
    return prefix

p = trace_build_prefix([-2, 0, 3, -5, 2, -1])

trạng thái đầu: [0, 0, 0, 0, 0, 0, 0]
i=0, x=-2: prefix[1] =  0 + -2 = -2 -> [0, -2, 0, 0, 0, 0, 0]
i=1, x= 0: prefix[2] = -2 +  0 = -2 -> [0, -2, -2, 0, 0, 0, 0]
i=2, x= 3: prefix[3] = -2 +  3 =  1 -> [0, -2, -2, 1, 0, 0, 0]
i=3, x=-5: prefix[4] =  1 + -5 = -4 -> [0, -2, -2, 1, -4, 0, 0]
i=4, x= 2: prefix[5] = -4 +  2 = -2 -> [0, -2, -2, 1, -4, -2, 0]
i=5, x=-1: prefix[6] = -2 + -1 = -3 -> [0, -2, -2, 1, -4, -2, -3]


### 7.2. Trace truy vấn — đối chiếu với cộng trực tiếp

Mỗi dòng in: đoạn cần cộng, tổng trực tiếp, và công thức prefix. Hai cột phải trùng nhau.

In [14]:
def trace_queries(nums: list[int], queries) -> None:
    prefix = [0] * (len(nums) + 1)
    for i, x in enumerate(nums):
        prefix[i + 1] = prefix[i] + x
    for left, right in queries:
        segment = nums[left:right + 1]
        direct = sum(segment)
        fast = prefix[right + 1] - prefix[left]
        print(f"[{left},{right}] đoạn={segment} | trực tiếp={direct:>2} | prefix[{right + 1}]-prefix[{left}]={prefix[right + 1]}-({prefix[left]})={fast:>2} | khớp={direct == fast}")

trace_queries([-2, 0, 3, -5, 2, -1], [(0, 2), (2, 5), (0, 5), (0, 0), (5, 5), (1, 3)])

[0,2] đoạn=[-2, 0, 3] | trực tiếp= 1 | prefix[3]-prefix[0]=1-(0)= 1 | khớp=True
[2,5] đoạn=[3, -5, 2, -1] | trực tiếp=-1 | prefix[6]-prefix[2]=-3-(-2)=-1 | khớp=True
[0,5] đoạn=[-2, 0, 3, -5, 2, -1] | trực tiếp=-3 | prefix[6]-prefix[0]=-3-(0)=-3 | khớp=True
[0,0] đoạn=[-2] | trực tiếp=-2 | prefix[1]-prefix[0]=-2-(0)=-2 | khớp=True
[5,5] đoạn=[-1] | trực tiếp=-1 | prefix[6]-prefix[5]=-3-(-2)=-1 | khớp=True
[1,3] đoạn=[0, 3, -5] | trực tiếp=-2 | prefix[4]-prefix[1]=-4-(-2)=-2 | khớp=True


## 8. Các lời giải khác

### Cách 3 — `accumulate(nums, initial=0)`
Cùng ý tưởng, code gọn hơn.

In [15]:
from itertools import accumulate

class NumArrayAccumulate:
    def __init__(self, nums: list[int]):
        self.prefix = list(accumulate(nums, initial=0))

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]

obj = NumArrayAccumulate([-2, 0, 3, -5, 2, -1])
print(obj.sumRange(0, 2), obj.sumRange(2, 5), obj.sumRange(0, 5))

1 -1 -3


### Cách 4 — prefix "không có số 0 ở đầu" (inclusive prefix)

`pre[i] = nums[0] + ... + nums[i]`. Khi đó `sumRange = pre[right] - pre[left - 1]`, và phải **xử lý riêng `left = 0`**. Cách này đúng nhưng dễ sai biên hơn — lý do ta thích phần tử canh `0`.

In [16]:
class NumArrayInclusive:
    def __init__(self, nums: list[int]):
        self.pre = list(accumulate(nums))      # không có initial

    def sumRange(self, left: int, right: int) -> int:
        if left == 0:
            return self.pre[right]
        return self.pre[right] - self.pre[left - 1]

obj = NumArrayInclusive([-2, 0, 3, -5, 2, -1])
print(obj.pre)
print(obj.sumRange(0, 2), obj.sumRange(2, 5), obj.sumRange(0, 5))

[-2, -2, 1, -4, -2, -3]
1 -1 -3


## 9. Code nộp LeetCode

### Cách chính — prefix sum với phần tử canh 0

In [17]:
class NumArray:

    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + x

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]


# Your NumArray object will be instantiated and called as such:
obj = NumArray([-2, 0, 3, -5, 2, -1])
print(obj.sumRange(0, 2))

1


### Cách nộp thay thế — brute force (chạy được với constraints này nhưng chậm hơn nhiều)

Với `n, q <= 10^4`, brute force có thể vẫn qua nhờ `sum(slice)` viết bằng C, nhưng đó không phải bài học của đề.

In [18]:
class NumArraySlice:
    def __init__(self, nums: list[int]):
        self.nums = nums

    def sumRange(self, left: int, right: int) -> int:
        return sum(self.nums[left:right + 1])

print(NumArraySlice([-2, 0, 3, -5, 2, -1]).sumRange(2, 5))

-1


## 10. Vì sao đúng?

**Invariant khi dựng:** sau bước `i` của vòng lặp, `prefix[i + 1] = nums[0] + ... + nums[i]`.
- Đúng với `prefix[0] = 0` (tổng rỗng).
- Nếu `prefix[i]` đúng thì `prefix[i + 1] = prefix[i] + nums[i]` cũng đúng (cộng thêm đúng một phần tử).

**Công thức truy vấn:** `prefix[right + 1]` gồm `nums[0..right]`, `prefix[left]` gồm `nums[0..left-1]`. Vì `left <= right`, phần chung `nums[0..left-1]` bị trừ hết, còn lại đúng `nums[left..right]`.

Mảng **bất biến** nên prefix tính một lần vẫn đúng mãi. (Nếu đề có cập nhật phần tử — bài 307 — prefix sẽ lỗi thời, cần cấu trúc khác như Fenwick Tree.)

## 11. Time / Space Complexity và so sánh

Gọi `n = len(nums)`, `q` = số truy vấn.

| Cách | `__init__` | `sumRange` | Tổng cho q truy vấn | Bộ nhớ phụ |
|---|---|---|---|---|
| Brute force vòng lặp | `O(1)` | `O(n)` | `O(q·n)` | `O(1)` |
| Brute force `sum(slice)` | `O(1)` | `O(n)` (+ copy slice) | `O(q·n)` | `O(n)` tạm |
| Tính mọi cặp | `O(n²)` | `O(1)` | `O(n² + q)` | `O(n²)` |
| **Prefix sum** | **`O(n)`** | **`O(1)`** | **`O(n + q)`** | **`O(n)`** |
| `accumulate` | `O(n)` | `O(1)` | `O(n + q)` | `O(n)` |

Pattern **"tiền xử lý một lần, truy vấn nhiều lần"**: chấp nhận `O(n)` lúc đầu để mỗi truy vấn rẻ.

In [19]:
import time
for cls in [NumArrayBrute, NumArray]:
    t0 = time.perf_counter()
    obj = cls(nums_big)
    total = sum(obj.sumRange(l, r) for l, r in queries)
    print(f"{cls.__name__:<14} tổng kiểm tra = {total:>12} | thời gian ≈ {time.perf_counter() - t0:.4f}s")

NumArrayBrute  tổng kiểm tra =   -584538531 | thời gian ≈ 0.0432s
NumArray       tổng kiểm tra =   -584538531 | thời gian ≈ 0.0008s


## 12. Bộ test — mẫu đề, biên, đối chiếu

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

commands = ["NumArray", "sumRange", "sumRange", "sumRange"]
args = [[[-2, 0, 3, -5, 2, -1]], [0, 2], [2, 5], [0, 5]]
check("ví dụ đề", run_ops(NumArray, commands, args), [None, 1, -1, -3])

one = NumArray([7])
check("mảng 1 phần tử", one.sumRange(0, 0), 7)
obj = NumArray([-2, 0, 3, -5, 2, -1])
check("left = right ở đầu", obj.sumRange(0, 0), -2)
check("left = right ở cuối", obj.sumRange(5, 5), -1)
check("đoạn giữa", obj.sumRange(1, 3), -2)
zeros = NumArray([0, 0, 0])
check("toàn 0", zeros.sumRange(0, 2), 0)
big = NumArray([10**5] * 10**4)
check("giá trị lớn nhất, n lớn nhất", big.sumRange(0, 10**4 - 1), 10**9)
neg = NumArray([-10**5] * 10**4)
check("giá trị nhỏ nhất", neg.sumRange(0, 10**4 - 1), -10**9)

PASS ví dụ đề: [None, 1, -1, -3]
PASS mảng 1 phần tử: 7
PASS left = right ở đầu: -2
PASS left = right ở cuối: -1
PASS đoạn giữa: -2
PASS toàn 0: 0
PASS giá trị lớn nhất, n lớn nhất: 1000000000
PASS giá trị nhỏ nhất: -1000000000


In [21]:
random.seed(42)
classes = [NumArrayBrute, NumArrayPrefix, NumArrayAccumulate, NumArrayInclusive, NumArraySlice, NumArray]
for _ in range(200):
    n = random.randint(1, 30)
    arr = [random.randint(-10, 10) for _ in range(n)]
    objs = [cls(arr) for cls in classes]
    for _ in range(20):
        l = random.randrange(n)
        r = random.randrange(l, n)
        expected = sum(arr[l:r + 1])
        for o in objs:
            assert o.sumRange(l, r) == expected, (type(o).__name__, arr, l, r)
small = NumArrayAllPairs([3, -1, 4, 1, -5])
assert all(small.sumRange(l, r) == sum([3, -1, 4, 1, -5][l:r + 1]) for l in range(5) for r in range(l, 5))
print("200 mảng ngẫu nhiên × 20 truy vấn: mọi cách đều khớp")

200 mảng ngẫu nhiên × 20 truy vấn: mọi cách đều khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1: Mảng prefix dài `n` và công thức `prefix[right] - prefix[left]`
Thiếu phần tử canh → lệch một vị trí: đoạn bị **mất `nums[right]`** hoặc **thừa/thiếu** ở đầu.

In [22]:
class NumArrayOffByOne:
    def __init__(self, nums):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + x

    def sumRange(self, left, right):
        return self.prefix[right] - self.prefix[left]      # SAI: phải là right + 1

wrong = NumArrayOffByOne([-2, 0, 3, -5, 2, -1])
print("sai :", wrong.sumRange(0, 2), "| đúng:", NumArray([-2, 0, 3, -5, 2, -1]).sumRange(0, 2))

sai : -2 | đúng: 1


### Lỗi 2: Tính prefix trong `sumRange` thay vì `__init__`
Đúng kết quả nhưng mỗi truy vấn lại `O(n)` — mất toàn bộ lợi ích.

### Lỗi 3: Dùng biến cục bộ thay vì `self.`
Viết `prefix = [...]` trong `__init__` (thiếu `self.`) → `sumRange` không thấy biến → `NameError`/`AttributeError`.

In [23]:
class NumArrayNoSelf:
    def __init__(self, nums):
        prefix = list(accumulate(nums, initial=0))    # SAI: biến cục bộ, mất sau khi __init__ kết thúc

    def sumRange(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

try:
    NumArrayNoSelf([1, 2, 3]).sumRange(0, 1)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NumArrayNoSelf' object has no attribute 'prefix'


### Lỗi 4: Slice thiếu `+ 1`
`sum(nums[left:right])` bỏ mất `nums[right]` vì slice không gồm đầu phải.

### Lỗi 5: Lưu tham chiếu tới `nums` rồi người khác sửa `nums`
Cách prefix không bị ảnh hưởng vì đã tính xong; cách brute force lưu `self.nums = nums` sẽ thấy thay đổi. Đề nói mảng bất biến nên không sao, nhưng nên biết: muốn an toàn thì `self.nums = list(nums)` (sao chép).

In [24]:
arr = [1, 2, 3]
print("sai :", sum(arr[0:2]), "| đúng:", sum(arr[0:3]))   # sumRange(0, 2) phải là 6
brute = NumArrayBrute(arr)
fast = NumArray(arr)
arr[0] = 100       # sửa mảng gốc bên ngoài
print("brute thấy thay đổi:", brute.sumRange(0, 2), "| prefix giữ giá trị cũ:", fast.sumRange(0, 2))

sai : 3 | đúng: 6
brute thấy thay đổi: 105 | prefix giữ giá trị cũ: 6


## 14. Mẹo nhận dạng pattern

- **"Nhiều truy vấn tổng đoạn `[l, r]`"** trên mảng **không đổi** → **prefix sum**.
- Công thức vàng: `sum(l..r) = prefix[r + 1] - prefix[l]` với `prefix[0] = 0`.
- Đề có chữ **Design / class / nhiều lần gọi** → hỏi: phần nào tính **một lần** trong `__init__`, phần nào tính **mỗi lần gọi**?
- Mở rộng: tổng đoạn = 0? tổng đoạn = k? → prefix sum + hash map (bài 560). Ma trận 2D → prefix 2D (bài 304).

## 15. Biến thể

### Biến thể 1 — Range Sum Query 2D (LeetCode 304)
`P[i+1][j+1] = M[i][j] + P[i][j+1] + P[i+1][j] - P[i][j]`, và tổng hình chữ nhật dùng **bao hàm – loại trừ**.

In [25]:
class NumMatrix:
    def __init__(self, matrix: list[list[int]]):
        rows, cols = len(matrix), len(matrix[0])
        self.P = [[0] * (cols + 1) for _ in range(rows + 1)]
        for i in range(rows):
            for j in range(cols):
                self.P[i + 1][j + 1] = matrix[i][j] + self.P[i][j + 1] + self.P[i + 1][j] - self.P[i][j]

    def sumRegion(self, r1: int, c1: int, r2: int, c2: int) -> int:
        P = self.P
        return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

m = NumMatrix([[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]])
print(m.sumRegion(2, 1, 4, 3), m.sumRegion(1, 1, 2, 2), m.sumRegion(1, 2, 2, 4))   # 8 11 12

8 11 12


### Biến thể 2 — đếm số đoạn con có tổng bằng `k` (LeetCode 560)
`sum(l..r) = k` ⇔ `prefix[r+1] - prefix[l] = k` ⇔ `prefix[l] = prefix[r+1] - k`. Đếm bằng dict tần suất các prefix đã thấy.

### Biến thể 3 — trung bình đoạn
`(prefix[r+1] - prefix[l]) / (r - l + 1)`.

In [26]:
def subarray_sum_equals_k(nums: list[int], k: int) -> int:
    seen = {0: 1}          # prefix 0 đã xuất hiện 1 lần (đoạn bắt đầu từ chỉ số 0)
    running = 0
    count = 0
    for x in nums:
        running += x
        count += seen.get(running - k, 0)
        seen[running] = seen.get(running, 0) + 1
    return count

print(subarray_sum_equals_k([1, 1, 1], 2), subarray_sum_equals_k([1, 2, 3], 3))   # 2 2

2 2


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Với `nums = [4, -1, 2, 7]`, viết mảng `prefix` (dài 5) và tính `sumRange(1, 3)` bằng công thức.

### Bài 2 — Dự đoán output
`list(accumulate([1, 2, 3], initial=0))` in ra gì? `[0] * 3` in ra gì?

### Bài 3 — Điền chỗ trống

In [27]:
class ExerciseNumArray:
    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        # TODO: điền prefix[i + 1] = prefix[i] + nums[i]

    def sumRange(self, left: int, right: int) -> int:
        # TODO: một phép trừ
        pass

# o = ExerciseNumArray([-2, 0, 3, -5, 2, -1])
# assert (o.sumRange(0, 2), o.sumRange(2, 5), o.sumRange(0, 5)) == (1, -1, -3)

### Bài 4 — Tự viết: tổng của `k` phần tử cuối
Viết `exercise_last_k_sums(nums, k)` trả về list tổng của mọi cửa sổ dài `k` (sliding window) bằng prefix sum.

In [28]:
def exercise_last_k_sums(nums: list[int], k: int) -> list[int]:
    # TODO: dùng prefix, cửa sổ [i, i + k - 1]
    pass

# assert exercise_last_k_sums([1, 2, 3, 4, 5], 2) == [3, 5, 7, 9]
# assert exercise_last_k_sums([5], 1) == [5]

### Bài 5 — Biến thể: Find Pivot Index (LeetCode 724)
Trả về chỉ số `i` nhỏ nhất mà tổng bên trái `i` bằng tổng bên phải `i` (không tính `nums[i]`), hoặc `-1`.

In [29]:
def exercise_pivot_index(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_pivot_index([1, 7, 3, 6, 5, 6]) == 3
# assert exercise_pivot_index([1, 2, 3]) == -1
# assert exercise_pivot_index([2, 1, -1]) == 0

### Bài 6 — Mini challenge: tổng đoạn của các số chẵn
Thiết kế `ExerciseEvenSum(nums)` với `sumEven(left, right)` trả về tổng **các phần tử chẵn** trong đoạn, mỗi truy vấn `O(1)`.

In [30]:
class ExerciseEvenSum:
    def __init__(self, nums: list[int]):
        # TODO: prefix chỉ cộng phần tử chẵn
        pass

    def sumEven(self, left: int, right: int) -> int:
        # TODO
        pass

# e = ExerciseEvenSum([1, 2, 3, 4, 5, 6])
# assert e.sumEven(0, 5) == 12 and e.sumEven(2, 3) == 4 and e.sumEven(0, 0) == 0

## 17. Gợi ý 3 mức

**Bài 1** — (1) Bắt đầu `prefix[0] = 0`. (2) Cộng dồn từng phần tử. (3) `sumRange(1,3) = prefix[4] - prefix[1]`.

**Bài 2** — (1) `initial=0` thêm 0 ở đầu. (2) Phép `*` trên list lặp lại phần tử. (3) Đếm độ dài kết quả.

**Bài 3** — (1) Vòng `for i, x in enumerate(nums)`. (2) Nhớ `self.prefix`. (3) `return self.prefix[right + 1] - self.prefix[left]`.

**Bài 4** — (1) Cửa sổ bắt đầu ở `i` kết thúc ở `i + k - 1`. (2) `i` chạy `0..n-k`. (3) `prefix[i + k] - prefix[i]`.

**Bài 5** — (1) Tổng trái = `prefix[i]`. (2) Tổng phải = `total - prefix[i] - nums[i]`. (3) Duyệt `i` từ trái, trả về `i` đầu tiên thỏa.

**Bài 6** — (1) Prefix với `x if x % 2 == 0 else 0`. (2) Công thức truy vấn y hệt. (3) Lưu `self.prefix`.

## 18. Đáp án tham khảo

**Bài 1:** `prefix = [0, 4, 3, 5, 12]`; `sumRange(1, 3) = prefix[4] - prefix[1] = 12 - 4 = 8` (đúng: `-1 + 2 + 7 = 8`).

**Bài 2:** `[0, 1, 3, 6]` và `[0, 0, 0]`.

In [31]:
class AnswerNumArray:
    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + x

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]

def answer_last_k_sums(nums: list[int], k: int) -> list[int]:
    prefix = list(accumulate(nums, initial=0))
    return [prefix[i + k] - prefix[i] for i in range(len(nums) - k + 1)]

def answer_pivot_index(nums: list[int]) -> int:
    total = sum(nums)
    left_sum = 0
    for i, x in enumerate(nums):
        if left_sum == total - left_sum - x:
            return i
        left_sum += x
    return -1

class AnswerEvenSum:
    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + (x if x % 2 == 0 else 0)

    def sumEven(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]

assert list(accumulate([4, -1, 2, 7], initial=0)) == [0, 4, 3, 5, 12]
assert AnswerNumArray([4, -1, 2, 7]).sumRange(1, 3) == 8
assert list(accumulate([1, 2, 3], initial=0)) == [0, 1, 3, 6] and [0] * 3 == [0, 0, 0]
o = AnswerNumArray([-2, 0, 3, -5, 2, -1])
assert (o.sumRange(0, 2), o.sumRange(2, 5), o.sumRange(0, 5)) == (1, -1, -3)
assert answer_last_k_sums([1, 2, 3, 4, 5], 2) == [3, 5, 7, 9] and answer_last_k_sums([5], 1) == [5]
assert answer_pivot_index([1, 7, 3, 6, 5, 6]) == 3
assert answer_pivot_index([1, 2, 3]) == -1
assert answer_pivot_index([2, 1, -1]) == 0
e = AnswerEvenSum([1, 2, 3, 4, 5, 6])
assert e.sumEven(0, 5) == 12 and e.sumEven(2, 3) == 4 and e.sumEven(0, 0) == 0
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class NumArray:
    def __init__(self, nums: list[int]):
        self.prefix = [0] * (len(nums) + 1)
        for i, x in enumerate(nums):
            self.prefix[i + 1] = self.prefix[i] + x

    def sumRange(self, left: int, right: int) -> int:
        return self.prefix[right + 1] - self.prefix[left]
```

**Công thức nhớ:** `prefix` dài `n + 1`, `prefix[0] = 0`, `sum(l..r) = prefix[r + 1] - prefix[l]`.

**Checklist trước khi submit:**
- [ ] Prefix được tính **một lần** trong `__init__`, lưu vào `self.`?
- [ ] Mảng prefix dài `n + 1` với `prefix[0] = 0`?
- [ ] Công thức dùng `right + 1` (đoạn bao gồm `right`)?
- [ ] Thử `left = 0`, `left = right`, mảng 1 phần tử?
- [ ] Tên class `NumArray`, method `sumRange(self, left, right)` đúng chữ ký?